In [ ]:
```xml
<VSCode.Cell language="markdown">
# Ch.7 — Networking & Load Balancing (Azure Supplement)

**Cloud platform:** Azure Application Gateway (L7 load balancer)

This notebook demonstrates:
- Azure Application Gateway setup
- Backend pool configuration
- Health probes
- SSL offloading
- Azure Load Balancer (L4 alternative)

In [ ]:
# Cell 1: Azure Credentials
# IMPORTANT: These will be stripped by pre-push hook

AZURE_SUBSCRIPTION_ID = ""  # Your Azure subscription ID
AZURE_RESOURCE_GROUP = ""   # Resource group name
AZURE_LOCATION = "eastus"   # Azure region
AZURE_VNET_NAME = "lb-vnet"
AZURE_SUBNET_NAME = "backend-subnet"
AZURE_APP_GATEWAY_NAME = "app-gateway-lb"

# Verify Azure CLI is installed
import subprocess
import json

try:
    result = subprocess.run(["az", "--version"], capture_output=True, text=True, check=True)
    print("✓ Azure CLI installed")
except Exception as e:
    print("✗ Azure CLI not found. Install from: https://aka.ms/installazurecliwindows")
    raise

# Login check
try:
    result = subprocess.run(["az", "account", "show"], capture_output=True, text=True, check=True)
    account = json.loads(result.stdout)
    print(f"✓ Logged in as: {account['user']['name']}")
    print(f"  Subscription: {account['name']}")
except:
    print("⚠ Not logged in. Run: az login")
    raise

print("\n⚠ REMINDER: Fill in credentials above before running subsequent cells")

In [ ]:
# Cell 2: Azure Application Gateway (L7 Load Balancer)

print("=== Azure Application Gateway Overview ===\n")

print("Application Gateway is a Layer 7 (HTTP/HTTPS) load balancer that provides:")
print("- URL path-based routing (e.g., /api/* → backend1, /admin/* → backend2)")
print("- SSL termination (HTTPS → HTTP to backends)")
print("- Web Application Firewall (WAF) protection")
print("- Autoscaling and multi-region support")
print("- Health probes with custom endpoints")

print("\n=== Architecture ===")
print("Client → Application Gateway (public IP)")
print("         ↓")
print("  Backend Pool (3 VMs or Container Instances)")
print("         ├── backend1 (10.0.1.4)")
print("         ├── backend2 (10.0.1.5)")
print("         └── backend3 (10.0.1.6)")

# Create resource group
if AZURE_RESOURCE_GROUP:
    print(f"\nCreating resource group: {AZURE_RESOURCE_GROUP}")
    subprocess.run([
        "az", "group", "create",
        "--name", AZURE_RESOURCE_GROUP,
        "--location", AZURE_LOCATION
    ], check=True)
    print("✓ Resource group created")
else:
    print("\n⚠ Set AZURE_RESOURCE_GROUP in Cell 1")

# Create virtual network
print(f"\nCreating virtual network: {AZURE_VNET_NAME}")
subprocess.run([
    "az", "network", "vnet", "create",
    "--resource-group", AZURE_RESOURCE_GROUP,
    "--name", AZURE_VNET_NAME,
    "--address-prefix", "10.0.0.0/16",
    "--subnet-name", AZURE_SUBNET_NAME,
    "--subnet-prefix", "10.0.1.0/24"
], check=True)

# Create subnet for Application Gateway
print("Creating Application Gateway subnet...")
subprocess.run([
    "az", "network", "vnet", "subnet", "create",
    "--resource-group", AZURE_RESOURCE_GROUP,
    "--vnet-name", AZURE_VNET_NAME,
    "--name", "appgw-subnet",
    "--address-prefix", "10.0.2.0/24"
], check=True)

print("\n✓ Network infrastructure created")
print(f"  VNet: {AZURE_VNET_NAME} (10.0.0.0/16)")
print(f"  Backend subnet: {AZURE_SUBNET_NAME} (10.0.1.0/24)")
print(f"  App Gateway subnet: appgw-subnet (10.0.2.0/24)")

In [ ]:
# Cell 3: Backend Pool Configuration (Deploy Container Instances)

print("=== Creating Backend Pool (3 Flask Containers) ===\n")

# Deploy 3 Azure Container Instances as backends
backends = []

for i in range(1, 4):
    backend_name = f"backend{i}"
    print(f"Deploying {backend_name}...")
    
    result = subprocess.run([
        "az", "container", "create",
        "--resource-group", AZURE_RESOURCE_GROUP,
        "--name", backend_name,
        "--image", "your-acr.azurecr.io/flask-backend:latest",  # Replace with your image
        "--vnet", AZURE_VNET_NAME,
        "--subnet", AZURE_SUBNET_NAME,
        "--cpu", "1",
        "--memory", "1",
        "--ports", "5000",
        "--environment-variables", f"BACKEND_ID={backend_name}",
        "--query", "ipAddress.ip",
        "--output", "tsv"
    ], capture_output=True, text=True, check=True)
    
    backend_ip = result.stdout.strip()
    backends.append({"name": backend_name, "ip": backend_ip})
    print(f"✓ {backend_name} deployed at {backend_ip}")

print("\n✓ Backend pool ready:")
for backend in backends:
    print(f"  - {backend['name']}: {backend['ip']}:5000")

# Save backend IPs for Application Gateway configuration
BACKEND_IPS = [b["ip"] for b in backends]

print("\n=== Backend Pool Configuration ===")
print("Backend addresses will be added to Application Gateway in next cell")

In [ ]:
# Cell 4: Configure Application Gateway with Health Probes

print("=== Creating Application Gateway ===\n")

# Create public IP for Application Gateway
print("Creating public IP...")
subprocess.run([
    "az", "network", "public-ip", "create",
    "--resource-group", AZURE_RESOURCE_GROUP,
    "--name", f"{AZURE_APP_GATEWAY_NAME}-pip",
    "--allocation-method", "Static",
    "--sku", "Standard"
], check=True)

# Create Application Gateway
print(f"\nCreating Application Gateway: {AZURE_APP_GATEWAY_NAME}")
print("(This may take 5-10 minutes...)")

# Build backend addresses parameter
backend_servers = " ".join(BACKEND_IPS)

subprocess.run([
    "az", "network", "application-gateway", "create",
    "--resource-group", AZURE_RESOURCE_GROUP,
    "--name", AZURE_APP_GATEWAY_NAME,
    "--location", AZURE_LOCATION,
    "--vnet-name", AZURE_VNET_NAME,
    "--subnet", "appgw-subnet",
    "--capacity", "2",  # Number of instances
    "--sku", "Standard_v2",
    "--http-settings-cookie-based-affinity", "Disabled",
    "--frontend-port", "80",
    "--http-settings-port", "5000",
    "--http-settings-protocol", "Http",
    "--public-ip-address", f"{AZURE_APP_GATEWAY_NAME}-pip",
    "--servers", *BACKEND_IPS
], check=True)

print("\n✓ Application Gateway created")

# Configure custom health probe
print("\nConfiguring health probe (/health endpoint)...")
subprocess.run([
    "az", "network", "application-gateway", "probe", "create",
    "--resource-group", AZURE_RESOURCE_GROUP,
    "--gateway-name", AZURE_APP_GATEWAY_NAME,
    "--name", "health-probe",
    "--protocol", "Http",
    "--host", "127.0.0.1",
    "--path", "/health",
    "--interval", "30",  # Probe interval in seconds
    "--timeout", "30",   # Timeout in seconds
    "--threshold", "3"   # Unhealthy threshold (failures before marking down)
], check=True)

# Update backend HTTP settings to use custom probe
subprocess.run([
    "az", "network", "application-gateway", "http-settings", "update",
    "--resource-group", AZURE_RESOURCE_GROUP,
    "--gateway-name", AZURE_APP_GATEWAY_NAME,
    "--name", "appGatewayBackendHttpSettings",
    "--probe", "health-probe"
], check=True)

print("\n✓ Health probe configured:")
print("  - Endpoint: /health")
print("  - Interval: 30 seconds")
print("  - Timeout: 30 seconds")
print("  - Unhealthy threshold: 3 failures")

# Get Application Gateway public IP
result = subprocess.run([
    "az", "network", "public-ip", "show",
    "--resource-group", AZURE_RESOURCE_GROUP,
    "--name", f"{AZURE_APP_GATEWAY_NAME}-pip",
    "--query", "ipAddress",
    "--output", "tsv"
], capture_output=True, text=True, check=True)

gateway_ip = result.stdout.strip()
print(f"\n✓ Application Gateway ready at: http://{gateway_ip}")
print("\nTest with: curl http://{gateway_ip}/api/payment")

In [ ]:
# Cell 5: SSL Offloading (HTTPS Termination)

print("=== SSL Offloading Configuration ===\n")

print("Application Gateway can terminate SSL/TLS and forward HTTP to backends:")
print("1. Upload SSL certificate (PFX format)")
print("2. Create HTTPS listener (port 443)")
print("3. Configure backend pool (HTTP, port 5000)")
print("4. Client → Gateway (HTTPS) → Backend (HTTP)")

print("\n=== Step 1: Create Self-Signed Certificate (Demo) ===")

# Generate self-signed certificate
subprocess.run([
    "openssl", "req", "-x509", "-nodes", "-days", "365",
    "-newkey", "rsa:2048",
    "-keyout", "appgw-key.pem",
    "-out", "appgw-cert.pem",
    "-subj", "/CN=example.com"
], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Convert to PFX (required by Application Gateway)
subprocess.run([
    "openssl", "pkcs12", "-export",
    "-out", "appgw-cert.pfx",
    "-inkey", "appgw-key.pem",
    "-in", "appgw-cert.pem",
    "-passout", "pass:Azure123"  # Demo password
], check=True)

print("✓ Certificate created: appgw-cert.pfx (password: Azure123)")

print("\n=== Step 2: Configure HTTPS Listener ===")

# Add HTTPS frontend port
subprocess.run([
    "az", "network", "application-gateway", "frontend-port", "create",
    "--resource-group", AZURE_RESOURCE_GROUP,
    "--gateway-name", AZURE_APP_GATEWAY_NAME,
    "--name", "https-port",
    "--port", "443"
], check=True)

# Upload SSL certificate
subprocess.run([
    "az", "network", "application-gateway", "ssl-cert", "create",
    "--resource-group", AZURE_RESOURCE_GROUP,
    "--gateway-name", AZURE_APP_GATEWAY_NAME,
    "--name", "appgw-ssl-cert",
    "--cert-file", "appgw-cert.pfx",
    "--cert-password", "Azure123"
], check=True)

# Create HTTPS listener
subprocess.run([
    "az", "network", "application-gateway", "http-listener", "create",
    "--resource-group", AZURE_RESOURCE_GROUP,
    "--gateway-name", AZURE_APP_GATEWAY_NAME,
    "--name", "https-listener",
    "--frontend-port", "https-port",
    "--ssl-cert", "appgw-ssl-cert"
], check=True)

# Create routing rule
subprocess.run([
    "az", "network", "application-gateway", "rule", "create",
    "--resource-group", AZURE_RESOURCE_GROUP,
    "--gateway-name", AZURE_APP_GATEWAY_NAME,
    "--name", "https-rule",
    "--http-listener", "https-listener",
    "--rule-type", "Basic",
    "--address-pool", "appGatewayBackendPool",
    "--http-settings", "appGatewayBackendHttpSettings"
], check=True)

print("\n✓ HTTPS listener configured")
print("  - Frontend: HTTPS (port 443)")
print("  - Backend: HTTP (port 5000)")
print("  - Certificate: appgw-ssl-cert")

print("\nTest with: curl -k https://{gateway_ip}/api/payment")
print("(-k flag ignores self-signed certificate warning)")

In [ ]:
# Cell 6: Azure Load Balancer (L4 Alternative)

print("=== Azure Load Balancer (Layer 4) ===\n")

print("Application Gateway (L7) vs Load Balancer (L4):")
print()
print("| Feature                  | Application Gateway (L7) | Load Balancer (L4)     |")
print("|--------------------------|--------------------------|------------------------|")
print("| Layer                    | Application (HTTP/HTTPS) | Transport (TCP/UDP)    |")
print("| URL-based routing        | ✓ Yes                    | ✗ No                   |")
print("| SSL termination          | ✓ Yes                    | ✗ No                   |")
print("| WAF protection           | ✓ Yes                    | ✗ No                   |")
print("| Session affinity         | ✓ Cookie-based           | ✓ Source IP hash       |")
print("| Health probes            | ✓ HTTP/HTTPS             | ✓ TCP/HTTP             |")
print("| Cost                     | Higher ($$$)             | Lower ($$)             |")
print("| Use case                 | Web apps, APIs           | Non-HTTP (DB, game)    |")

print("\n=== When to Use Load Balancer ===")
print("- Non-HTTP traffic (e.g., PostgreSQL, Redis, game servers)")
print("- High throughput (millions of connections)")
print("- Lower cost for simple TCP load balancing")

print("\n=== Creating Azure Load Balancer (Demo) ===")

LB_NAME = "backend-lb"
LB_PIP_NAME = f"{LB_NAME}-pip"

# Create public IP
print("Creating public IP for Load Balancer...")
subprocess.run([
    "az", "network", "public-ip", "create",
    "--resource-group", AZURE_RESOURCE_GROUP,
    "--name", LB_PIP_NAME,
    "--sku", "Standard"
], check=True)

# Create Load Balancer
print("Creating Load Balancer...")
subprocess.run([
    "az", "network", "lb", "create",
    "--resource-group", AZURE_RESOURCE_GROUP,
    "--name", LB_NAME,
    "--sku", "Standard",
    "--public-ip-address", LB_PIP_NAME,
    "--frontend-ip-name", "lb-frontend",
    "--backend-pool-name", "lb-backend-pool"
], check=True)

# Create health probe (TCP)
print("Creating TCP health probe...")
subprocess.run([
    "az", "network", "lb", "probe", "create",
    "--resource-group", AZURE_RESOURCE_GROUP,
    "--lb-name", LB_NAME,
    "--name", "tcp-probe",
    "--protocol", "tcp",
    "--port", "5000",
    "--interval", "15",
    "--threshold", "2"
], check=True)

# Create load balancing rule
print("Creating load balancing rule...")
subprocess.run([
    "az", "network", "lb", "rule", "create",
    "--resource-group", AZURE_RESOURCE_GROUP,
    "--lb-name", LB_NAME,
    "--name", "lb-rule",
    "--protocol", "tcp",
    "--frontend-port", "80",
    "--backend-port", "5000",
    "--frontend-ip-name", "lb-frontend",
    "--backend-pool-name", "lb-backend-pool",
    "--probe-name", "tcp-probe"
], check=True)

print("\n✓ Azure Load Balancer created")
print(f"  Name: {LB_NAME}")
print("  Protocol: TCP")
print("  Frontend port: 80")
print("  Backend port: 5000")
print("  Health probe: TCP on port 5000")

# Get Load Balancer public IP
result = subprocess.run([
    "az", "network", "public-ip", "show",
    "--resource-group", AZURE_RESOURCE_GROUP,
    "--name", LB_PIP_NAME,
    "--query", "ipAddress",
    "--output", "tsv"
], capture_output=True, text=True, check=True)

lb_ip = result.stdout.strip()
print(f"\n✓ Load Balancer ready at: {lb_ip}")

print("\n\n=== Cleanup (Run when done) ===")
print(f"az group delete --name {AZURE_RESOURCE_GROUP} --yes --no-wait")
print("\n⚠ This will delete all resources (Application Gateway, Load Balancer, backends)")